<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_20_inheritance_polymorphism/note_lesson_20_inheritance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 20 — Наслідування, поліморфізм: таксі, кур'єр, самовивіз

Сервіс «Смачно + Таксі» тепер має кілька способів доставки: **таксі** (ціна за районом), **кур'єр-пішохід** (40 грн, лише Поділ), **самовивіз** (безкоштовно), а вночі — надбавка. Звіт для всіх однаковий: `fare()`, `eta()`, `describe()`.

Виконуй клітинки **зверху вниз**; перед **Прогнозом** спершу скажи, що буде. Теорія, діаграми класів і порівняння «ієрархія проти композиції» — у книзі: [Урок 20. Наслідування, поліморфізм](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_20/).

## 🔁 Пригадай (без підглядання)

1. Де Python шукає `first.FEES`, якщо в екземплярі його немає?
2. Що повертає `isinstance(True, int)`?
3. Що таке фабрика функцій?

<details>
<summary>Відповіді</summary>

1. У класі.
2. `True`: `bool` — нащадок `int`.
3. Функція, що створює й повертає налаштовану функцію.

</details>

## 1. Базовий клас і перший нащадок

In [ ]:
class Delivery:
    """Будь-яка доставка: номер замовлення, район, ціна, час."""

    kind = "Доставка"

    def __init__(self, order_id, district):
        self.order_id = order_id
        self.district = district

    def fare(self):
        raise NotImplementedError(f"{type(self).__name__} має визначити fare()")

    def eta(self):
        return 30

    def describe(self):
        return f"{self.kind} №{self.order_id}: {self.district}, {self.fare()} грн, ~{self.eta()} хв"

In [ ]:
class Pickup(Delivery):
    kind = "Самовивіз"

    def fare(self):
        return 0

    def eta(self):
        return 15


pickup = Pickup(3, "кафе")
print(pickup.describe())
print(isinstance(pickup, Pickup), isinstance(pickup, Delivery))

**Прогноз:** у якому порядку Python шукатиме метод для `Pickup`?

In [ ]:
print([cls.__name__ for cls in Pickup.mro()])

<details>
<summary>Відповідь</summary>

`Pickup → Delivery → object`. `describe` знайдено в `Delivery`, але `self.fare()` усередині знову шукається з `Pickup`.

</details>

## 2. `super()`: розширити, а не замінити

In [ ]:
class TaxiDelivery(Delivery):
    kind = "Таксі"
    FEES = {"Поділ": 60, "Оболонь": 80, "Печерськ": 90}

    def __init__(self, order_id, district, driver):
        super().__init__(order_id, district)
        self.driver = driver

    def fare(self):
        return self.FEES[self.district]

    def eta(self):
        return 25

    def describe(self):
        return super().describe() + f", водій {self.driver}"


taxi = TaxiDelivery(1, "Оболонь", "D-3")
print(taxi.describe())
print(taxi.__dict__)

## 3. Поліморфізм

In [ ]:
class CourierDelivery(Delivery):
    kind = "Кур'єр"
    AREA = {"Поділ"}

    def __init__(self, order_id, district):
        if district not in self.AREA:
            raise ValueError(f"кур'єр не ходить у район {district}")
        super().__init__(order_id, district)

    def fare(self):
        return 40

    def eta(self):
        return 45

In [ ]:
deliveries = [TaxiDelivery(1, "Оболонь", "D-3"), CourierDelivery(2, "Поділ"), Pickup(3, "кафе")]

for delivery in deliveries:
    print(delivery.describe())
print("Разом за доставку:", sum(delivery.fare() for delivery in deliveries), "грн")

### Качина типізація

**Прогноз:** чи спрацює звіт з об'єктом, що **не** наслідує `Delivery`?

In [ ]:
class PartnerCourier:
    """Клас з бібліотеки партнера — не наслідує Delivery."""

    def __init__(self, order_id, price):
        self.order_id = order_id
        self.price = price

    def fare(self):
        return self.price

    def describe(self):
        return f"Партнер №{self.order_id}: {self.price} грн"


mixed = deliveries + [PartnerCourier(4, 95)]
print(sum(delivery.fare() for delivery in mixed))
print(mixed[-1].describe(), isinstance(mixed[-1], Delivery))

<details>
<summary>Відповідь</summary>

Так: звіт лише викликає `fare()` і `describe()`. `isinstance(..., Delivery)` при цьому — `False`.

</details>

## 4. Міксин і множинне наслідування

**Прогноз:** скільки коштуватиме нічне таксі на Печерськ (90 грн × 1.3)? Яким буде MRO?

In [ ]:
class NightMixin:
    """Нічна надбавка 30 % до ціни будь-якої доставки."""

    def fare(self):
        return round(super().fare() * 1.3)

    def describe(self):
        return super().describe() + " (ніч)"


class NightTaxi(NightMixin, TaxiDelivery):
    pass


night = NightTaxi(5, "Печерськ", "D-2")
print(night.describe())
print([cls.__name__ for cls in NightTaxi.mro()])

<details>
<summary>Відповідь</summary>

117 грн. MRO: `NightTaxi → NightMixin → TaxiDelivery → Delivery → object`; `super().fare()` у міксині веде до `TaxiDelivery`.

</details>

## 🐛 Вправа 1. Обірвана естафета

Міксин `LoggedMixin` пише в журнал про створення доставки, але його `__init__` не викликає `super().__init__()`. **Прогноз:** що впаде? Виправ міксин.

In [ ]:
LOG = []


class LoggedMixin:
    def __init__(self, *args):
        LOG.append(f"створено {type(self).__name__}")
        # YOUR CODE HERE — чого бракує?
        # BEGIN SOLUTION
        super().__init__(*args)
        # END SOLUTION


class LoggedCourier(LoggedMixin, CourierDelivery):
    pass


courier = LoggedCourier(8, "Поділ")
print(courier.describe(), LOG)
assert courier.describe() == "Кур'єр №8: Поділ, 40 грн, ~45 хв"
assert LOG == ["створено LoggedCourier"]
print("✅ Вправа 1 пройдена")

## 5. Пастки: несумісні сигнатури й нащадок, що ламає батька

In [ ]:
KINDS = {"taxi": TaxiDelivery, "courier": CourierDelivery, "pickup": Pickup}

for name, cls in KINDS.items():
    try:
        print(cls(7, "Поділ").describe())
    except TypeError as error:
        print(name, "→", error)

In [ ]:
class FriendlyPickup(Pickup):
    def fare(self):
        return "безкоштовно"


try:
    print(sum(delivery.fare() for delivery in deliveries + [FriendlyPickup(6, "кафе")]))
except TypeError as error:
    print(error)

<details>
<summary>Що тут не так?</summary>

`FriendlyPickup` порушує обіцянку батька «`fare()` повертає число» — принцип підстановки Лісков (LSP). Будь-який код, що рахує суму, падає.

</details>

## 6. Розібраний приклад: звіт, що не знає типів доставок

In [ ]:
def delivery_report(deliveries):
    by_kind = {}
    for delivery in deliveries:
        by_kind.setdefault(delivery.kind, []).append(delivery)
    lines = []
    for kind, group in by_kind.items():
        total = sum(delivery.fare() for delivery in group)
        lines.append(f"{kind}: {len(group)} шт., {total} грн")
    fastest = min(deliveries, key=lambda delivery: delivery.eta())
    lines.append(f"Найшвидша: {fastest.describe()}")
    return "\n".join(lines)


shift = [TaxiDelivery(1, "Оболонь", "D-3"), CourierDelivery(2, "Поділ"), Pickup(3, "кафе"),
         NightTaxi(4, "Печерськ", "D-2"), TaxiDelivery(5, "Поділ", "D-1")]
print(delivery_report(shift))

## 🛠 Вправа 2. Доставка дроном

`DroneDelivery(Delivery)`: `kind = "Дрон"`, 120 грн, 10 хвилин, лише Оболонь (інакше `ValueError` у конструкторі). `delivery_report` не змінюй.

In [ ]:
class DroneDelivery(Delivery):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    kind = "Дрон"
    AREA = {"Оболонь"}

    def __init__(self, order_id, district):
        if district not in self.AREA:
            raise ValueError(f"дрон не літає в район {district}")
        super().__init__(order_id, district)

    def fare(self):
        return 120

    def eta(self):
        return 10
    # END SOLUTION


class NightDrone(NightMixin, DroneDelivery):
    pass


text = delivery_report(shift + [DroneDelivery(6, "Оболонь")])
print(text)
assert "Дрон: 1 шт., 120 грн" in text
assert text.endswith("Найшвидша: Дрон №6: Оболонь, 120 грн, ~10 хв")
assert NightDrone(7, "Оболонь").fare() == 156
try:
    DroneDelivery(9, "Поділ")
except ValueError:
    pass
else:
    raise AssertionError("дрон на Поділ — ValueError")
print("✅ Вправа 2 пройдена")

## 7. Архітектура: ієрархія чи композиція?

Три способи × день/ніч × з промокодом / без — в ієрархії це **12 класів**. Композиція: доставка **має** правило ціни — функцію «район → ціна», а правила комбінуються, як конвеєр з уроку 18.

| | Ієрархія класів | Композиція |
|---|---|---|
| Новий спосіб | новий клас | нова функція-правило |
| Комбінації ознак | вибух класів / міксини | `night(promo(taxi_pricing))` |
| Коли доречно | стабільне «є» | «має», змінні правила |

## 🛠 Вправа 3. Те саме композицією

Один клас `ComposedDelivery(order_id, district, pricing, kind)` без підкласів; функції `taxi_pricing`, `courier_pricing` (лише Поділ, інакше `ValueError`) і фабрика `night(pricing)`, що додає 30 %.

In [ ]:
FEES = {"Поділ": 60, "Оболонь": 80, "Печерськ": 90}


class ComposedDelivery:
    def __init__(self, order_id, district, pricing, kind):
        self.order_id = order_id
        self.district = district
        self.pricing = pricing
        self.kind = kind

    def fare(self):
        return self.pricing(self.district)


# YOUR CODE HERE: taxi_pricing, courier_pricing, night
# BEGIN SOLUTION
def taxi_pricing(district):
    return FEES[district]


def courier_pricing(district):
    if district != "Поділ":
        raise ValueError(f"кур'єр не ходить у район {district}")
    return 40


def night(pricing):
    def priced(district):
        return round(pricing(district) * 1.3)
    return priced
# END SOLUTION


assert ComposedDelivery(1, "Оболонь", taxi_pricing, "Таксі").fare() == 80
assert ComposedDelivery(4, "Печерськ", night(taxi_pricing), "Таксі").fare() == 117
assert ComposedDelivery(2, "Поділ", night(courier_pricing), "Кур'єр").fare() == 52
assert ComposedDelivery(5, "Печерськ", night(taxi_pricing), "Таксі").fare() == NightTaxi(5, "Печерськ", "D-2").fare()
print("✅ Вправа 3 пройдена: композиція дає ті самі ціни")

## ✅ Самоперевірка

1. Звідки в `Pickup` береться `describe` і чому `self.fare()` в ньому повертає 0?
2. Що станеться без `super().__init__` у `TaxiDelivery`?
3. Чому `super().fare()` у `NightMixin` веде до `TaxiDelivery`?
4. Чим поганий `FriendlyPickup`?
5. Коли наслідування, а коли композиція?

<details>
<summary>Відповіді</summary>

1. З `Delivery` за MRO; `self` — `Pickup`, тож `fare` шукається з `Pickup`.
2. Не буде `order_id` і `district` — `AttributeError` в `describe`.
3. `super()` — наступний клас у MRO **екземпляра**.
4. Порушує LSP: `fare()` має повертати число.
5. «Є» і стабільно — наслідування; «має», змінні правила й комбінації — композиція.

</details>

### Шпаргалка

```python
class Taxi(Delivery):                        # нащадок
    def __init__(self, order_id, district, driver):
        super().__init__(order_id, district)  # спершу батько
        self.driver = driver
    def fare(self):                          # перевизначення
        return ...
    def describe(self):
        return super().describe() + "..."    # розширення

Taxi.mro()                                   # порядок пошуку методів
class NightTaxi(NightMixin, Taxi): ...       # міксин першим; кожен викликає super()

for d in deliveries:                         # поліморфізм — без if за типом
    d.fare()

Delivery(order_id, district, pricing=night(taxi_pricing))   # композиція
```

## Далі

- Практикум на реальних даних — [`lab_lesson_20_titanic_inheritance.ipynb`](https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_20_inheritance_polymorphism/lab_lesson_20_titanic_inheritance.ipynb): ієрархія пасажирів «Титаніка».
- **Урок 21 — Інкапсуляція, область видимості**: як не дати коду ззовні записати в доставку від'ємну ціну.